In [ ]:
# ======================
# 🔹 Load Two Models
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

latent_dim = 64
gen1 = QuantumGenerator(latent_dim=latent_dim, n_qubits=6, q_depth=4, out_channels=1).to(device)
gen2 = QuantumGenerator(latent_dim=latent_dim, n_qubits=6, q_depth=4, out_channels=1).to(device)

# 🔥 Update these with your checkpoint paths
gen1.load_state_dict(torch.load("/content/drive/MyDrive/QGAN_Outputs/fashion_20260109_171124/gen_epoch20.pt", map_location=device))
gen2.load_state_dict(torch.load("/content/drive/MyDrive/QGAN_Outputs/fashion_20260109_171124/gen_epoch16.pt", map_location=device))

gen1.eval()
gen2.eval()

# ======================
# 🔹 Generate Outputs
# ======================
z = torch.randn(64, latent_dim, device=device)

fake1 = gen1(z)
fake2 = gen2(z)

# (1) Image Averaging
ensemble_avg = 0.5 * (fake1 + fake2)

# (2) Latent-Level Mixing (randomly pick per sample)
mask = torch.randint(0, 2, (z.size(0), 1, 1, 1), device=device).float()
ensemble_mix = mask * fake1 + (1 - mask) * fake2

# (3) Sample Pooling (just concatenate, used for evaluation not images)
pooled_fake = torch.cat([fake1, fake2], dim=0)

# ======================
# 🔹 Save Samples
# ======================
outdir = "/content/drive/MyDrive/QGAN_Outputs/Ensemble_Test"
os.makedirs(outdir, exist_ok=True)

save_image_grid(fake1.cpu(), os.path.join(outdir, "gen1_samples.png"))
save_image_grid(fake2.cpu(), os.path.join(outdir, "gen2_samples.png"))
save_image_grid(ensemble_avg.cpu(), os.path.join(outdir, "ensemble_avg.png"))
save_image_grid(ensemble_mix.cpu(), os.path.join(outdir, "ensemble_mix.png"))

print(f"✅ Samples saved in {outdir}")

# ======================
# 🔹 Compute Scores
# ======================
dl = make_dataloader("fashion", batch_size=512)
real, _ = next(iter(dl))
real = real[:512].to(device)
real_eval = (real + 1) / 2

fake1_eval = (fake1 + 1) / 2
fake2_eval = (fake2 + 1) / 2
avg_eval   = (ensemble_avg + 1) / 2
mix_eval   = (ensemble_mix + 1) / 2
pooled_eval = (pooled_fake + 1) / 2

fid1 = compute_fid(fake1_eval, real_eval, device)
fid2 = compute_fid(fake2_eval, real_eval, device)
fid_avg = compute_fid(avg_eval, real_eval, device)
fid_mix = compute_fid(mix_eval, real_eval, device)
fid_pooled = compute_fid(pooled_eval, real_eval, device)

jsd1 = compute_jsd(fake1_eval, real_eval)
jsd2 = compute_jsd(fake2_eval, real_eval)
jsd_avg = compute_jsd(avg_eval, real_eval)
jsd_mix = compute_jsd(mix_eval, real_eval)
jsd_pooled = compute_jsd(pooled_eval, real_eval)

print("🔹 Results:")
print(f"FID gen1: {fid1:.3f}, gen2: {fid2:.3f}, avg: {fid_avg:.3f}, mix: {fid_mix:.3f}, pooled: {fid_pooled:.3f}")
print(f"JSD gen1: {jsd1:.6f}, gen2: {jsd2:.6f}, avg: {jsd_avg:.6f}, mix: {jsd_mix:.6f}, pooled: {jsd_pooled:.6f}")

✅ Samples saved in /content/drive/MyDrive/QGAN_Outputs/ensemble_test
🔹 Results:
FID gen1: 123.566, gen2: 140.087, avg: 146.032, mix: 120.148, pooled: 102.827
JSD gen1: 0.009867, gen2: 0.008496, avg: 0.038399, mix: 0.006880, pooled: 0.007347
